## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-17-20-51-24 +0000,nyt,Man Indicted in Bomb Threat at Illinois Ethano...,https://www.nytimes.com/2026/11/17/us/peoria-i...
1,2026-11-17-20-50-00 +0000,wsj,Oil Futures Post Modest Gains,https://www.wsj.com/finance/commodities-future...
2,2026-11-17-20-49-15 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/17/techno...
3,2026-11-17-20-46-45 +0000,nypost,US launches attack on Iran’s Qeshm Island in r...,https://nypost.com/2026/11/17/world-news/us-st...
4,2026-11-17-20-44-54 +0000,bbc,American soldier arrested by UK police on susp...,https://www.bbc.co.uk/news/articles/c6x284ez38...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2539/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
75,trump,54
95,nvidia,27
349,china,21
290,huang,19
367,will,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
243,2026-11-17-12-57-00 +0000,wsj,Nvidia formed a board committee to oversee nat...,https://www.wsj.com/politics/national-security...,139
90,2026-11-17-18-34-00 +0000,wsj,A lawyer for Fed governor Lisa Cook said she w...,https://www.wsj.com/politics/policy/feds-lisa-...,138
306,2026-11-17-10-17-11 +0000,nypost,Trump greeted with ‘YMCA’ in Philippines to ki...,https://nypost.com/2026/11/17/us-news/trump-ar...,124
457,2026-11-16-21-07-23 +0000,nyt,Trump Administration Moves to Close A.I. Chip ...,https://www.nytimes.com/2026/11/16/us/politics...,119
361,2026-11-17-02-37-53 +0000,bbc,Nvidia dismisses 'Bring Jensen Back' campaign ...,https://www.bbc.co.uk/news/articles/cry49plv8v...,116


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
243,139,2026-11-17-12-57-00 +0000,wsj,Nvidia formed a board committee to oversee nat...,https://www.wsj.com/politics/national-security...
90,138,2026-11-17-18-34-00 +0000,wsj,A lawyer for Fed governor Lisa Cook said she w...,https://www.wsj.com/politics/policy/feds-lisa-...
217,95,2026-11-17-14-25-45 +0000,bbc,How new US chip restrictions on South East Asi...,https://www.bbc.co.uk/news/articles/cwq6e2qgvm...
306,60,2026-11-17-10-17-11 +0000,nypost,Trump greeted with ‘YMCA’ in Philippines to ki...,https://nypost.com/2026/11/17/us-news/trump-ar...
109,57,2026-11-17-18-04-54 +0000,latimes,You were caught up in the FAA airport chaos. A...,https://www.latimes.com/california/story/2026-...
309,54,2026-11-17-10-09-28 +0000,nypost,Russia says Ukraine struck its bridge to Crime...,https://nypost.com/2026/11/17/world-news/russi...
262,48,2026-11-17-11-53-00 +0000,wsj,Chris Van Hollen’s challenge of Chuck Schumer ...,https://www.wsj.com/politics/policy/van-hollen...
28,48,2026-11-17-20-14-01 +0000,latimes,Trump says report of Iran bomb plot against D....,https://www.latimes.com/politics/story/2026-11...
193,47,2026-11-17-15-35-04 +0000,nypost,Clarence Thomas calls foul on California rulin...,https://nypost.com/2026/11/17/us-news/clarence...
233,47,2026-11-17-13-31-58 +0000,nypost,DHS Secretary Markwayne Mullin suggests AI cou...,https://nypost.com/2026/11/17/us-news/dhs-secr...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
